<a href="https://colab.research.google.com/github/rohanmyers/queue-viability/blob/main/notebooks/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geopandas pyarrow openpyxl

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

DRIVE_ROOT = Path("/content/drive/MyDrive/queue-viability")
RAW = DRIVE_ROOT / "data" / "raw"
INTERIM = DRIVE_ROOT / "data" / "interim"
PROCESSED = DRIVE_ROOT / "data" / "processed"
FIGURES = DRIVE_ROOT / "reports" / "figures"

gen = pd.read_parquet(INTERIM / "generation_queue.parquet")
print(f"Loaded {len(gen):,} rows")

Mounted at /content/drive
Loaded 33,566 rows


In [ ]:
# Y label: 1 if operational, 0 if withdrawn
gen['y'] = np.where(
    gen['q_status'] == 'operational', 1,
    np.where(gen['q_status'] == 'withdrawn', 0, np.nan)
)

# Mark training-eligible rows
gen['is_trainable'] = (
    gen['y'].notna() &
    (gen['q_year'] <= 2019)
)

# Mark scoring-eligible rows (the live regulator-facing output)
gen['is_scoring'] = gen['q_status'].isin(['active', 'suspended'])

print(f"Trainable: {gen['is_trainable'].sum():,}")
print(f"Scoring-eligible: {gen['is_scoring'].sum():,}")

Trainable: 15,662
Scoring-eligible: 10,976


In [ ]:
# Total capacity
gen['mw_total'] = gen[['mw1', 'mw2', 'mw3']].sum(axis=1, min_count=1)

# Bucket rare fuel types — anything below 200 trainable rows becomes "Other_combined"
fuel_counts = gen.loc[gen['is_trainable'], 'type_clean'].value_counts()
common_fuels = fuel_counts[fuel_counts >= 200].index
gen['fuel_bucketed'] = gen['type_clean'].where(
    gen['type_clean'].isin(common_fuels),
    'Other_combined'
)

# Capacity log-transform (long tail — model will benefit)
# Replace any mw_total values <= -1 with NaN before applying log1p to avoid RuntimeWarning
gen['mw_total'] = gen['mw_total'].mask(gen['mw_total'] <= -1)
gen['log_mw'] = np.log1p(gen['mw_total'])

# Service type (ERIS/NRIS) — already clean
# Region — already clean
# Project type (Generation vs Surplus) — already clean

print("Direct features ready:")
print(gen[['fuel_bucketed', 'region', 'service', 'project_type']].nunique())

Direct features ready:
fuel_bucketed    9
region           9
service          4
project_type     2
dtype: int64


In [ ]:
gen.IA_status_clean.unique()

array(['Withdrawn', 'Operational', 'In Progress (unknown study)',
       'IA Executed', 'Construction', 'Suspended', 'Cluster Study',
       'System Impact Study', 'Facility Study', None, 'IA Pending',
       'Feasibility Study', 'Not Started', 'Combined'], dtype=object)

In [ ]:
# Map IA_status_clean to ordinal phase
# Order reflects how far along a project is in the interconnection process

phase_order = {
    'Not Started': 0,
    'Cluster Study': 1,
    'Feasibility Study': 2,
    'System Impact Study': 3,
    'Facility Study': 4,
    'IA Pending': 5,
    'IA Executed': 6,
    'Construction': 7,
    'Operational': 6,  # imputed: operational projects passed through IA Executed
    'Withdrawn': np.nan,
    'Suspended': np.nan,
    'In Progress (unknown study)': np.nan,
    'Combined': np.nan,
}


gen['phase_ordinal'] = gen['IA_status_clean'].map(phase_order)

print("Trainable rows with phase_ordinal:")
print(gen.loc[gen['is_trainable'], 'phase_ordinal'].describe())

Trainable rows with phase_ordinal:
count    8437.000000
mean        4.091028
std         1.647235
min         0.000000
25%         3.000000
50%         3.000000
75%         6.000000
max         6.000000
Name: phase_ordinal, dtype: float64


In [ ]:
pd.crosstab(gen['q_status'], gen['IA_status_clean'])

IA_status_clean,Cluster Study,Combined,Construction,Facility Study,Feasibility Study,IA Executed,IA Pending,In Progress (unknown study),Not Started,Operational,Suspended,System Impact Study,Withdrawn
q_status,,,,,,,,,,,,,
active,1030,0,45,1646,78,1884,95,2555,261,14,105,1049,1
operational,8,1,0,167,17,1751,34,19,0,1021,4,290,0
suspended,0,0,0,26,1,153,0,4,0,0,215,19,0
unknown,0,0,0,1,0,0,0,0,0,0,0,0,0
withdrawn,552,0,0,794,1631,996,2,2083,15,0,16,3193,6334


In [ ]:
# Queue vintage — when did this project enter the queue?
gen['q_year'] = pd.to_numeric(gen['q_year'], errors='coerce')

# Time between request and proposed online date (developer's stated timeline)
# Both q_date and prop_date are known at request time — no leakage
gen['q_date'] = pd.to_datetime(gen['q_date'], errors='coerce')
gen['prop_date'] = pd.to_datetime(gen['prop_date'], errors='coerce')

gen['proposed_dev_years'] = (
    (gen['prop_date'] - gen['q_date']).dt.days / 365.25
).clip(lower=0, upper=15)  # cap unreasonable values

In [ ]:
# Build a per-developer historical completion rate, time-aware
gen_sorted = gen.sort_values('q_date').copy()

# For each row, look at all that developer's prior projects with known outcomes
# (resolved before this project's q_date)
def compute_developer_history(df):
    """For each row, compute the developer's completion rate among
    projects that resolved before this row's queue date."""
    df = df.sort_values('q_date').copy()
    df['dev_prior_n'] = 0
    df['dev_prior_completion_rate'] = np.nan

    for dev, group in df.groupby('developer'):
        if pd.isna(dev) or dev == 'nan':
            continue
        group = group.sort_values('q_date')
        for idx, row in group.iterrows():
            # Prior projects = same developer, q_date earlier than this one,
            # and outcome (op/wd) is known
            prior = group[
                (group['q_date'] < row['q_date']) &
                (group['y'].notna())
            ]
            n = len(prior)
            df.at[idx, 'dev_prior_n'] = n
            if n >= 3:  # require at least 3 prior projects for a meaningful rate
                df.at[idx, 'dev_prior_completion_rate'] = prior['y'].mean()

    return df

gen = compute_developer_history(gen)

print(f"Rows with usable dev history (n>=3 prior): "
      f"{gen['dev_prior_completion_rate'].notna().sum():,}")
print(f"Mean completion rate among developers w/ history: "
      f"{gen['dev_prior_completion_rate'].mean():.2%}")

Rows with usable dev history (n>=3 prior): 1,530
Mean completion rate among developers w/ history: 28.87%


In [ ]:
# Distribution of dev_prior_n among rows that have it
print("Dev prior project counts (among rows with n>=3):")
print(gen.loc[gen['dev_prior_n'] >= 3, 'dev_prior_n'].describe())

# Top developers by row count
print("\nTop 15 developers by trainable row count:")
trainable = gen[gen['is_trainable']]
print(trainable['developer'].value_counts().head(15))

# Completion rate by top developers
top_devs = trainable['developer'].value_counts().head(15).index
completion_by_dev = (
    trainable[trainable['developer'].isin(top_devs)]
    .groupby('developer')['y']
    .agg(['mean', 'count'])
    .sort_values('count', ascending=False)
)
print("\nCompletion rate among top 15 developers:")
print(completion_by_dev)

Dev prior project counts (among rows with n>=3):
count    1530.000000
mean       58.656209
std        58.652512
min         3.000000
25%         7.000000
50%        33.000000
75%       103.000000
max       214.000000
Name: dev_prior_n, dtype: float64

Top 15 developers by trainable row count:
developer
Masked                                 173
FPL                                     93
Duke Energy Florida, LLC                89
PacifiCorp Commercial & Trading         39
Duke Energy Progress, LLC               30
PacifiCorp Energy                       23
SCPSA                                   22
EDF Renewables Development, Inc.        20
Nextera                                 15
sPower Development Company, LLC         15
Cypress Creek Renewables                14
Noble Environmental Power, LLC          12
Basin Electric                          12
Black Hills Electric Generation LLC     12
Eon                                     11
Name: count, dtype: int64

Completion rate among to

In [ ]:
# Replace "Masked" with NaN before computing developer history
gen['developer'] = gen['developer'].replace('Masked', np.nan)
# Rerun compute_developer_history
gen = compute_developer_history(gen)

# Recheck
print(f"Rows with usable dev history (n>=3 prior): "
      f"{gen['dev_prior_completion_rate'].notna().sum():,}")
print(f"Mean completion rate among developers w/ history: "
      f"{gen['dev_prior_completion_rate'].mean():.2%}")

Rows with usable dev history (n>=3 prior): 1,300
Mean completion rate among developers w/ history: 33.98%


In [ ]:
trainable = gen[gen['is_trainable']]
print("Top 15 developers (should not include 'Masked'):")
print(trainable['developer'].value_counts().head(15))

Top 15 developers (should not include 'Masked'):
developer
FPL                                    93
Duke Energy Florida, LLC               89
PacifiCorp Commercial & Trading        39
Duke Energy Progress, LLC              30
PacifiCorp Energy                      23
SCPSA                                  22
EDF Renewables Development, Inc.       20
sPower Development Company, LLC        15
Nextera                                15
Cypress Creek Renewables               14
Black Hills Electric Generation LLC    12
Noble Environmental Power, LLC         12
Basin Electric                         12
Eon                                    11
OneEnergy Development, LLC             10
Name: count, dtype: int64


In [ ]:
# Cluster size — how many projects are in the same cluster?
gen['cluster_size'] = gen.groupby('cluster')['cluster'].transform('count')

# Queue activity at time of entry — how many other projects entered the same year/region?
gen['q_year_region_volume'] = (
    gen.groupby(['q_year', 'region'])['q_id'].transform('count')
)

In [ ]:
# Trim to features + label + identifiers
feature_cols = [
    'q_id', 'q_status', 'q_date', 'q_year', 'region', 'state', 'county',
    'fips_codes', 'utility', 'developer', 'fuel_bucketed', 'service',
    'project_type', 'mw_total', 'log_mw', 'proposed_dev_years',
    'dev_prior_n', 'dev_prior_completion_rate',
    'cluster_size', 'q_year_region_volume',
    'y', 'is_trainable', 'is_scoring'
]

features_v1 = gen[feature_cols].copy()

# Cast q_id to string in case the parquet issue resurfaces
features_v1['q_id'] = features_v1['q_id'].astype(str)

features_v1.to_parquet(INTERIM / "features_v1.parquet", index=False)
print(f"Saved {len(features_v1):,} rows × {len(feature_cols)} cols")

Saved 33,566 rows × 23 cols
